# 07 · GPS sintético e sistemas não lineares

**Objetivo pedagógico:** Construir o Jacobiano de pseudodistâncias, estimar viés de relógio e avaliar geometria e ruído.

**Luigi Scofano de Araujo · USP 13680334**

**Extensão nova desta edição:** não foi recuperado um notebook histórico de GPS. O roteiro conecta sistemas não lineares, mínimos quadrados e transformação de coordenadas usando dados sintéticos. Esta edição foi preparada em 2026. Não é uma cópia autenticada de uma entrega nem uma afirmação de conclusão das questões originais. Os arquivos recuperados estão em `acervo/`, com créditos e limitações em `docs/PROCEDENCIA.md`.

Execute as células em ordem, em kernel novo. Os dados são locais/sintéticos e as conclusões numéricas são verificadas por asserções; gráficos sozinhos não certificam convergência.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"numericos").is_dir() else Path.cwd().parent
if not (ROOT/"numericos").is_dir():
    raise RuntimeError("Execute a partir da raiz do repositório ou de notebooks/.")
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=7, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True})
FIGURAS = ROOT/"assets"/"figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)
def mostrar(nome):
    plt.tight_layout()
    plt.savefig(FIGURAS/(nome+".svg"), bbox_inches="tight", metadata={"Date": None})
    plt.show()
    plt.close("all")

## 1. Modelo simplificado
Uma pseudodistância é ρᵢ=||r−sᵢ||+b, com posição r em ECEF, satélite sᵢ e viés b em metros. Quatro incógnitas exigem ao menos quatro observações independentes. Usamos Gauss–Newton e mínimos quadrados com verificação do posto, sem formar JᵀJ.

**Todos os dados são sintéticos.** As coordenadas de referência foram escolhidas para o exemplo e não representam a localização do titular. Os satélites não são efemérides e não há filtro de visibilidade do horizonte.

In [2]:
from numericos.gps import geodesicas_para_ecef,ecef_para_geodesicas,estimar_posicao
rng=np.random.default_rng(13680334)
satellites=rng.normal(size=(10,3));satellites*=2.66e7/np.linalg.norm(satellites,axis=1)[:,None]
truth_lla=np.array([-20.,-45.,750.]);truth=geodesicas_para_ecef(*truth_lla)
clock_bias=12000.
rho=np.linalg.norm(truth-satellites,axis=1)+clock_bias
result=estimar_posicao(satellites,rho)
print("ECEF estimado (m):",result.ecef)
print("latitude, longitude, altitude:",ecef_para_geodesicas(result.ecef))
print("viés (m):",result.vies_metros,"iterações:",result.iteracoes,"cond(J):",result.condicao_jacobiana)
assert np.allclose(result.ecef,truth,atol=1e-5,rtol=0)
assert abs(result.vies_metros-clock_bias)<1e-5
assert np.allclose(ecef_para_geodesicas(result.ecef),truth_lla,atol=1e-5,rtol=0)

ECEF estimado (m): [ 4240194.9132064 -4240194.9132064 -2167953.3029363]
latitude, longitude, altitude: [-20. -45. 750.]
viés (m): 11999.999999999965 iterações: 5 cond(J): 3.0084355102893863


## 2. Ruído e excesso de observações
Com ruído, o resíduo não deve ser forçado a zero. O erro de posição é conhecido apenas porque o experimento tem uma referência sintética. Repetições usam um gerador local com seed fixa.

In [3]:
errors=[];residuals=[]
for _ in range(60):
    noisy=rho+rng.normal(0,3,len(rho))
    estimate=estimar_posicao(satellites,noisy)
    errors.append(np.linalg.norm(estimate.ecef-truth))
    residuals.append(np.linalg.norm(estimate.residuos))
print("erro mediano (m):",np.median(errors),"máximo (m):",np.max(errors))
plt.hist(errors,bins=12);plt.xlabel("erro 3D (m)");plt.ylabel("repetições")
mostrar("07_gps_erro")

erro mediano (m): 2.9288864500092897 máximo (m): 5.971718274381678


## 3. Geometria degenerada
Repetir o mesmo satélite não cria quatro restrições independentes. A falha precisa ser sinalizada, não escondida num vetor de saída não inicializado.

In [4]:
try:
    estimar_posicao(np.tile([2e7,0,0],(4,1)),np.full(4,2e7))
except np.linalg.LinAlgError as error:
    print("Falha esperada:",error)
else:
    raise AssertionError("Geometria degenerada aceita indevidamente")

Falha esperada: Geometria insuficiente para estimar quatro incógnitas.


## Limites e atividade
Este exemplo omite correções de relógio dos satélites, propagação atmosférica, rotação terrestre durante o trânsito, efemérides, relatividade e multicaminho. Não é um receptor GNSS nem serve para navegação real.

Os parâmetros do elipsoide WGS84 são a=6378137 m e 1/f=298.257223563; a referência é a NGA, listada em `docs/REFERENCIAS.md`. Varie a distribuição dos satélites e compare cond(J), resíduo e erro da posição.

## Figuras desta execução

![07_gps_erro](../assets/figuras/07_gps_erro.svg)